## Open Government Data, provided by **OpenDataZurich**
*Autogenerated Python starter code for data set with identifier* **geo_wasserrettung**

## Dataset
# **Wasserrettung**

## Description

Schutz &amp; Rettung Zürich hat für das Dispogebiet der Einsatzleitzentrale Rettungskonzepte erstellt.
Die vorliegenden Geodaten bilden die Grundlage für die Erarbeitung und Anwendung dieser Konzepte. Zudem werden sie für die Alarmierung und Disposition der Rettungskräfte benötigt.
Youtube: https://www.youtube.com/watch?v=xl3zgnHF6Oo

**Zweck**: Geobasisdaten als Grundlage für die Gewässerrettungskonzepte, die von Schutz &amp; Rettung Zürich in Zusammenarbeit mit den zuständigen Feuerwehren erarbeitet wurden.

**Genereller Hinweis zum Geodatensatz:**

Es handelt sich hierbei um einen Geodatensatz, welcher aus mehreren Geodatenelmenten besteht. Diese Geodatenelemente werden hier nicht im Detail beschrieben. Sie finden jedoch sämtliche Informationen - wie beispielsweise die Attributbeschreibungen - [«**Wasserrettung»** auf Geocat.ch](https://www.geocat.ch/geonetwork/srv/ger/catalog.search#/metadata/19ef2f91-64d5-43f0-8f97-26881c7cca1e).
			   

**Informationen zum Datensatz:**

Die vorliegenden Geodaten sollen den beteiligten Organisationen als Grundlage für die Weiterverarbeitung sowie für die Erstellung und Pflege ihrer eigenen Konzepte zur Verfügung stehen.

**Datenerfassung:**

Datenerfassung in QGIS

**Datengrundlage:**

Datenerfassung ab Amtlicher Vermessung Bodenbedeckung Gewässer, swisstopo TLM Daten und OpenStreetMap (OSM)

**Geodatenelemente:**

Die vollständige Metadatenbeschreibung der Geodatenelemente finden Sie ebenfalls unter [«**Wasserrettung»** auf Geocat.ch](https://www.geocat.ch/geonetwork/srv/ger/md.viewer#/full_view/19ef2f91-64d5-43f0-8f97-26881c7cca1e).

***sektoren***: 

Fliessgewässer mit Abschnitten inklusive Untersektoren


***uebergaenge***: 

Übergänge wie Brücken, Kraftwerke / Wehre 




**Statisches Vorschaubild:**

![BildText](https://www.gis.stadt-zuerich.ch/zueriplan_docs/geocat/19ef2f91-64d5-43f0-8f97-26881c7cca1e.png)



## Data set links

[Direct link by OpenDataZurich for dataset](https://data.stadt-zuerich.ch/dataset/geo_wasserrettung)

https://www.stadt-zuerich.ch/geodaten/download/Wasserrettung?format=geojson_link<br>


## Metadata
- **Publisher** `Einsatzplanung, Schutz & Rettung, Sicherheitsdepartement`
- **Maintainer** `Open Data Zürich`
- **Maintainer_email** `opendata@zuerich.ch`
- **Keywords** `Basis­karten,Umwelt,Ver­walt­ung`
- **Tags** `['feuerwehr', 'fliesswasser', 'geodaten', 'geoportal', 'gewasser', 'polizei', 'punktdaten', 'rettung', 'rettungsdienst', 'schweizerische-lebensrettungs-gesellschaft-slrg', 'stzh']`
- **Metadata_created** `2026-08-19T06:05:02.084600`
- **Metadata_modified** `2026-10-13T06:40:54.109534`


## Imports and helper functions

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd 

import json
import xml.etree.ElementTree as ET
import requests
import geopandas as gpd
import re

# Identifies requests from this notebook in the OpenDataZurich access logs.
USER_AGENT = "OpenDataZurich-StarterCode-Notebook/1.0 (lang=python; +https://github.com/opendatazurich/starter-code)"
HEADERS = {"User-Agent": USER_AGENT}

In [ ]:
# helper function for reading datasets with proper separator
def get_dataset(url):
    # get URL of WFS service
    dataset_identifier = re.search(r'\/([^\/\?]+)\?', url).group(1)
    url_geoportal = f"https://www.ogd.stadt-zuerich.ch/wfs/geoportal/{dataset_identifier}"
    print("Getting available layers from:", url_geoportal)
    
    # Parameter for GetCapabilities
    params = {
        "service": "WFS",
        "version": "1.1.0",
        "request": "GetCapabilities"
    }
    
    # send GetCapabilities
    response = requests.get(url_geoportal, params=params, headers=HEADERS)

    # parse XML answer
    root = ET.fromstring(response.content)

    # define Namespace 
    namespace = {'wfs': 'http://www.opengis.net/wfs'}
    
    # Extract available layers
    layers = [feature_type.find('wfs:Name', namespace).text for feature_type in root.findall('.//wfs:FeatureType', namespace)]
    
    print("Available layers:", layers)
    print("First layer is set as default. To choose another layer set it as typename in the get_dataset() function.")

    # set first layer as typename
    typename = layers[0]
    print("Chosen typename:", typename)

    # Parameter GetFeature request
    params = {
        "service": "WFS",
        "version": "1.1.0",
        "request": "GetFeature",
        "typename": typename,
        "outputFormat": "application/json"
    }
    
    # GetFeature request
    response = requests.get(url_geoportal, params=params, headers=HEADERS)

    # Load GeoJSON in GeoDataFrame
    gdf = gpd.read_file(json.dumps(response.json()))
    return gdf

## Load the data

In [ ]:
df = get_dataset('https://www.stadt-zuerich.ch/geodaten/download/Wasserrettung?format=geojson_link')

## Analyze the data

In [ ]:
df.plot()

In [ ]:
# drop columns that have no values
df.dropna(how='all', axis=1, inplace=True)

In [ ]:
print(f'The dataset has {df.shape[0]:,.0f} rows (observations) and {df.shape[1]:,.0f} columns (variables).')
print(f'There seem to be {df.duplicated().sum()} exact duplicates in the data.')

In [ ]:
df.info(memory_usage='deep', verbose=True)

In [ ]:
df.head()

In [ ]:
# display a small random sample transposed in order to see all variables
df.sample(3).T

In [ ]:
# describe non-numerical features
try:
    with pd.option_context('display.float_format', '{:,.2f}'.format):
        display(df.describe(exclude='number'))
except:
    print("No categorical data in dataset.")

In [ ]:
# describe numerical features
try:
    with pd.option_context('display.float_format', '{:,.2f}'.format):
        display(df.describe(include='number'))
except:
    print("No numerical data in dataset.")

In [ ]:
# Column types, completeness, and distinct non-missing values.
profile = pd.DataFrame(
    {
        "dtype": df.dtypes.astype(str),
        "missing": df.isna().sum(),
        "missing_pct": df.isna().mean().mul(100),
        "unique": df.nunique(dropna=True),
    }
).sort_values("missing_pct", ascending=False)
display(profile.round(1))

In [ ]:
# Show up to 20 columns with missing values; percentages use all rows.
missing = profile.loc[profile["missing"] > 0].head(20).rename_axis("column").reset_index()
if not missing.empty:
    fig, ax = plt.subplots(figsize=(8, max(2, len(missing) * 0.3)))
    ax.barh(missing["column"], missing["missing_pct"], color="steelblue")
    ax.set(xlim=(0, 100), xlabel="Missing values (%)", ylabel="", title="Missingness by column")
    fig.tight_layout()
    plt.show()
else:
    print("No missing values to plot (or the dataset has no rows).")

In [ ]:
# plot a histogram for each numerical feature
try:
    df.hist(bins=25, rwidth=.9)
    plt.tight_layout()
    plt.show()
except:
    print("No numerical data to plot.") 

In [ ]:
# continue your code here...

**Contact**: opendata@zuerich.ch